In [1]:
import os
import qsprpred
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd
from qsprpred.data.descriptors.sets import RDKitDescs

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
def load_datasets(path):
    dataset = QSPRDataset.fromTableFile(
    filename=path,
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
    )
    dataset.prepareDataset(
    feature_calculators=[MorganFP(radius=2, nBits=1024)],
    recalculate_features=True,
    shuffle=False
    )
    from qsprpred.data.descriptors.sets import RDKitDescs
    
    rdkit_descs = RDKitDescs()
    
    dataset.addDescriptors([rdkit_descs])
    
    dataset.descriptorSets
    return dataset
    

In [3]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizerFast, RobertaForMaskedLM, DataCollatorWithPadding
from sklearn.base import BaseEstimator, TransformerMixin

class SMILESDataset(Dataset):
    def __init__(self, smiles, tokenizer, max_len=128):
        self.smiles = smiles
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smile = self.smiles[idx]
        encoding = self.tokenizer(smile, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return encoding


class ChemBERTaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name="entropy/roberta_zinc_480m", max_len=128, batch_size=32, device=None):
        self.model_name = model_name
        self.max_len = max_len
        self.batch_size = batch_size
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = RobertaForMaskedLM.from_pretrained(self.model_name).to(self.device)
        self.tokenizer = RobertaTokenizerFast.from_pretrained(self.model_name, max_len=self.max_len)
        self.collator = DataCollatorWithPadding(self.tokenizer, padding=True, return_tensors='pt')
        self.embedding_dim = None  # bude nastaven po fit()

    def fit(self, X, y=None):
        # Zjistíme embedding dimenzi na prvním SMILES
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=1, collate_fn=self.collator)
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                embedding = outputs[1][-1]  # poslední hidden state
                self.embedding_dim = embedding.shape[-1]
                break
        return self

    def transform(self, X):
        self.model.eval()
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=self.batch_size, collate_fn=self.collator)
        embeddings_list = []

        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                full_embeddings = outputs[1][-1]
                embeddings = ((full_embeddings * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(-1).unsqueeze(-1))
                embeddings_list.append(embeddings)

        all_embeddings = torch.cat(embeddings_list, dim=0).cpu().numpy()
        column_names = [f"chemberta_{i}" for i in range(self.embedding_dim)]
        return pd.DataFrame(all_embeddings, columns=column_names)


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X1_all = load_datasets("CK1/data/ck1_train_1")

X2_all = load_datasets("CK1/data/ck1_val_1")

X3_all = load_datasets("CK1/data/ck1_test_1")

transformer = ChemBERTaTransformer()
X_train_emb = transformer.fit_transform(X1_all.df["Drug"])
X_val_emb = transformer.transform(X2_all.df["Drug"])
X_test_emb = transformer.transform(X3_all.df["Drug"])


/tmp/ipykernel_32073/195630644.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  smile = self.smiles[idx]


In [5]:
X1_all.df.Y.sum()/X1_all.df.Y.count()

0.1864754098360656

In [6]:
import pandas as pd
X1_all.X = X1_all.X.reset_index(drop=True)
X_train_emb = X_train_emb.reset_index(drop=True)
X1_all.X = pd.concat([X1_all.X, X_train_emb], axis = 1)

In [7]:
X2_all.X = X2_all.X.reset_index(drop=True)
X_val_emb = X_val_emb.reset_index(drop=True)
X2_all.X = pd.concat([X2_all.X, X_val_emb], axis = 1)
X3_all.X = X3_all.X.reset_index(drop=True)
X_test_emb = X_test_emb.reset_index(drop=True)
X3_all.X = pd.concat([X3_all.X, X_test_emb], axis = 1)

In [8]:
X1 = X1_all.X
y1 = X1_all.y
X2 = X2_all.X
y2 = X2_all.y
X3 = X3_all.X
y3 = X3_all.y

In [9]:
display(X1.shape)

(488, 2002)

In [10]:
imp_mean = SimpleImputer(missing_values=pd.NA, strategy='mean')
X1 = imp_mean.fit_transform(X1)
X2 = imp_mean.transform(X2)
X3 = imp_mean.transform(X3)
scaler = StandardScaler()
scaler.fit(X1)
X1 = scaler.transform(X1)
X2 = scaler.transform(X2)
X3 = scaler.transform(X3)

In [11]:
pd.DataFrame(X1).columns[pd.DataFrame(X1).isna().any()].tolist()



[]

In [12]:
from sklearn.decomposition import PCA
pca = PCA(n_components=10)
X1 = pca.fit_transform(X1)
X2 = pca.transform(X2)
X3 = pca.transform(X3)
display(X1)


array([[  2.5161142 ,  -6.1514845 ,  -8.366763  , ...,  -1.3118901 ,
         -0.6662839 ,   1.513191  ],
       [ -9.859417  , -11.0455475 ,   0.94790745, ...,  -0.6696343 ,
          0.695835  ,  -0.7949366 ],
       [-14.141907  ,   0.43424284,  -6.3157997 , ...,  -0.14369504,
          5.264172  ,   6.478469  ],
       ...,
       [ 14.9598055 ,  -5.998557  ,  -6.402905  , ...,   2.3849971 ,
         -4.536833  ,   4.0228906 ],
       [  0.87557787, -11.750225  ,  -9.756429  , ...,   0.9695275 ,
          8.374207  ,  -7.945045  ],
       [ -9.742928  , -10.71623   ,   1.2024305 , ...,   0.03662867,
          1.6882728 ,   0.32325163]], dtype=float32)

In [13]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.5, random_state=42)
X1, y1 = smote.fit_resample(X1, y1)
display(pd.DataFrame(X1))


,0,1,2,3,4,5,6,7,8,9
0,2.516114,-6.151484,-8.366763,-7.528070,-5.214782,3.237812,0.589255,-1.311890,-0.666284,1.513191
1,-9.859417,-11.045547,0.947907,7.023404,3.557904,-0.847088,16.151550,-0.669634,0.695835,-0.794937
2,-14.141907,0.434243,-6.315800,-3.726554,-0.599752,-2.600497,-2.427831,-0.143695,5.264172,6.478469
3,1.866866,-4.928757,-1.755384,3.479956,-3.932843,9.223029,1.024431,-2.455681,-5.013060,-6.554518
4,18.861055,10.574770,5.742424,2.776769,0.898590,-0.107555,11.001244,-8.188697,0.694418,-4.586987
...,...,...,...,...,...,...,...,...,...,...
590,-0.361225,-0.107655,-3.506367,-0.088767,16.778597,-4.424897,-13.856416,5.748622,7.335232,-1.304725
591,1.275800,-0.652349,-3.429613,-0.303002,16.270290,-3.513508,-14.506849,5.026931,7.291084,-0.767619
592,11.375954,-6.403468,-2.478024,1.294257,-2.773849,6.262298,-0.820110,3.818690,-5.169132,3.736483
593,-11.614032,8.930408,-3.423841,5.952575,-4.161908,-1.674558,1.227061,1.691786,13.959568,0.200364


In [14]:

# Přidejte cestu k vašemu lokálnímu repozitáři
import sys
import os

# Přidání cesty k lokálnímu repozitáři na začátek sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Zkontrolujte, zda je cesta v sys.path
print(sys.path)

from importlib import reload

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected
# Znovu načtěte modul, abyste zajistili, že je správně importován
reload(sys.modules['qsprpred.extra.gpu.models.neural_network'])

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected

os.chdir('/home/ubuntu/Bakalarka/QSPRpred')
print(os.getcwd())


import sys
import importlib.util

# Přidání cesty k repozitáři do sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Specifikujte cestu k souboru, který chcete importovat
module_path = '/home/ubuntu/Bakalarka/QSPRpred/qsprpred/extra/gpu/models/neural_network.py'
module_name = 'qsprpred.extra.gpu.models.neural_network'

# Načtěte modul z konkrétní cesty
spec = importlib.util.spec_from_file_location(module_name, module_path)
neural_network = importlib.util.module_from_spec(spec)
spec.loader.exec_module(neural_network)

# Nyní můžete používat třídu STFullyConnected
STFullyConnected = neural_network.STFullyConnected

['/home/ubuntu/Bakalarka/QSPRpred', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python311.zip', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/lib-dynload', '', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages']
/home/ubuntu/Bakalarka/QSPRpred
lol


In [15]:
from sklearn.model_selection import ParameterGrid
from torch.nn import functional as F
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, matthews_corrcoef
import pandas as pd

def test_fun(dic,  X_train, y_train, X_test, y_test) -> pd.DataFrame:
    param_grid_t = ParameterGrid(dic)
    i = 0
    val_f1_t = []
    val_acc_t = []
    val_mcc_t = []
    param_len_t = len(param_grid_t)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device used:", device)
    for param in param_grid_t:
        i += 1
        print(i, '/', param_len_t)
        model_sts_t = STFullyConnected(n_dim=X_train.shape[1],  # počet vstupních neuronů (počet deskriptorů)
        n_class=1,  # regresní úloha (1 výstup)
        gpus=[],
        device=device,
        is_reg=False, **param)
        model_sts_t.fit(X_train, y_train)
        res = model_sts_t.predict(X_test)
        res = res >0.5
        val_f1_t.append(f1_score(res, y_test))
        val_acc_t.append(accuracy_score(res, y_test))
        val_mcc_t.append(matthews_corrcoef(res, y_test))
        print(param)
        print(f1_score(res, y_test))
        print(accuracy_score(res, y_test))
        print(matthews_corrcoef(res, y_test))
    my_df = pd.DataFrame(param_grid_t)
    my_df["F1"] = val_f1_t
    my_df["Acc"] = val_acc_t
    my_df["MCC"] = val_mcc_t
    return my_df

In [16]:
import optuna
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef
import torch
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import confusion_matrix
def objective(trial, X_train, y_train, X_test, y_test):
    dropout_frac = trial.suggest_categorical("dropout_frac", [0, 0.1, 0.2, 0.4, 0.5, 0.6, 0.8, 0.9])
    patience = trial.suggest_categorical("patience", [10, 40, 75])
    tol = trial.suggest_categorical("tol", [1e-5, 1e-4, 1e-3, 1e-2, 0])
    weight_decay = trial.suggest_categorical("weight_decay", [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 0])
    n_epochs = trial.suggest_categorical("n_epochs", [200, 300, 500, 1000])
    batch_size = trial.suggest_categorical("batch_size", [1024, 512, 256, 128, 64])
    optimizer = trial.suggest_categorical("optimizer", ["optim.AdamW", "optim.RMSprop"])
    lr = trial.suggest_categorical("lr", [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6])
    neuron_layers_dict = {
    '[4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8]': [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8],
    '[2048, 1024, 512, 256, 128, 64, 32, 16, 8]': [2048, 1024, 512, 256, 128, 64, 32, 16, 8],
    '[4096, 1024, 256, 64, 8]': [4096, 1024, 256, 64, 8],
    '[4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8]': [4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8],
    '[4096, 2048, 1024, 512, 256, 512, 1024, 2048, 4096]': [4096, 2048, 1024, 512, 256, 512, 1024, 2048, 4096],
    '[200]': [200],
    '[2000]': [2000],
    '[2000, 1000]': [2000, 1000],
    '[2000, 1000, 500]': [2000, 1000, 500],
    '[1000, 50]': [1000, 50],
    '[4000, 2000]': [4000, 2000],
    '[4000, 2000, 1000, 500]': [4000, 2000, 1000, 500],
    '[4000, 2000, 2000, 500]': [4000, 2000, 2000, 500]
    }
    neuron_layers_size = trial.suggest_categorical("neuron_layers_size", list(neuron_layers_dict.keys()))
    opt = {"optim.AdamW": optim.AdamW,
           "optim.RMSprop": optim.RMSprop}
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(device)
    
    # Model
    model = STFullyConnected(
        n_dim=X_train.shape[1],
        n_class=1,
        gpus=[],
        device=device,
        is_reg=False,
        act_fun=F.selu,
        dropout_frac=dropout_frac,
        patience=patience,
        tol=tol,  # Opraveno: nyní používáme hodnotu z trial
        weight_decay=weight_decay,
        n_epochs=n_epochs,
        neuron_layers= neuron_layers_dict[neuron_layers_size],  # Použití neuron_layers_size
        batch_size=batch_size,
        optimizer=opt[optimizer],
        lr=lr,
        random_seed=69
    )
    # Trénink a predikce
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    preds_bin = preds > 0.5

    # Metiky
    f1 = f1_score(y_test, preds_bin)
    acc = accuracy_score(y_test, preds_bin)
    mcc = matthews_corrcoef(y_test, preds_bin)
    
    # Můžeš logovat i do trialu
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("acc", acc)
    display(confusion_matrix(y_test, preds_bin))
    return mcc  # maximalizujeme MCC


In [17]:
display(X1)

array([[  2.5161142 ,  -6.1514845 ,  -8.366763  , ...,  -1.3118901 ,
         -0.6662839 ,   1.513191  ],
       [ -9.859417  , -11.0455475 ,   0.94790745, ...,  -0.6696343 ,
          0.695835  ,  -0.7949366 ],
       [-14.141907  ,   0.43424284,  -6.3157997 , ...,  -0.14369504,
          5.264172  ,   6.478469  ],
       ...,
       [ 11.375954  ,  -6.403468  ,  -2.4780242 , ...,   3.8186903 ,
         -5.1691318 ,   3.7364826 ],
       [-11.614032  ,   8.930408  ,  -3.4238412 , ...,   1.6917858 ,
         13.959568  ,   0.20036384],
       [ -9.586387  , -10.694406  ,   1.8170375 , ...,  -0.9215643 ,
          0.8671848 ,   0.210772  ]], dtype=float32)

In [ ]:
study_3 = optuna.create_study(
    study_name="CK1_study_bert_bceloss_earlystopping_pca_3",  # jméno pro pozdější načtení
    direction="maximize",
    sampler=optuna.samplers.RandomSampler(),
    storage="sqlite:///optuna_results.db",
    load_if_exists=True  # pokud už existuje, nepřepíše ji
)

# Spusť optimalizaci
study_3.optimize(
    lambda trial: objective(trial, X1, y1, X2, y2),
    n_trials=5000
)
print("Best MCC:", study_3.best_value)
print("Best parameters:", study_3.best_params)

# Pokud chceš F1 a ACC u nejlepšího modelu:
print("Best F1:", study_3.best_trial.user_attrs["f1"])
print("Best ACC:", study_3.best_trial.user_attrs["acc"])

[I 2025-05-02 22:13:59,938] Using an existing study with name 'CK1_study_bert_bceloss_earlystopping_pca_3' instead of creating a new one.


cuda


array([[125,   0],
       [ 30,   0]])

[I 2025-05-02 22:14:46,101] Trial 45 finished with value: 0.0 and parameters: {'dropout_frac': 0.8, 'patience': 10, 'tol': 0.001, 'weight_decay': 0.1, 'n_epochs': 1000, 'batch_size': 1024, 'optimizer': 'optim.RMSprop', 'lr': 1e-05, 'neuron_layers_size': '[4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8]'}. Best is trial 41 with value: 0.4028916508145916.


cuda


...

[I 2025-05-02 22:18:17,520] Trial 51 finished with value: 0.274605531320799 and parameters: {'dropout_frac': 0.1, 'patience': 75, 'tol': 0.001, 'weight_decay': 0.001, 'n_epochs': 300, 'batch_size': 512, 'optimizer': 'optim.RMSprop', 'lr': 1e-05, 'neuron_layers_size': '[2000, 1000]'}. Best is trial 41 with value: 0.4028916508145916.


cuda


array([[125,   0],
       [ 30,   0]])

[I 2025-05-02 22:19:02,175] Trial 52 finished with value: 0.0 and parameters: {'dropout_frac': 0.4, 'patience': 75, 'tol': 0.01, 'weight_decay': 0.0001, 'n_epochs': 200, 'batch_size': 128, 'optimizer': 'optim.RMSprop', 'lr': 0.001, 'neuron_layers_size': '[4000, 2000]'}. Best is trial 41 with value: 0.4028916508145916.


cuda


In [ ]:
study_3.best_trial

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())


In [ ]:
import numpy as np
import pandas as pd

# Převod na DataFrame
X_final_train = pd.concat([pd.DataFrame(X1), pd.DataFrame(X2)], axis=0)
y_final_train = pd.concat([pd.DataFrame(y1), pd.DataFrame(y2)], axis=0)

# Pokud chceš mít je jako DataFrame s jedním sloupcem pro y


print(type(X_final_train))
print(type(y_final_train))

In [ ]:
model_final = STFullyConnected(
        n_dim=X1.shape[1],
        n_class=1,
        gpus=[],
        device="cuda",
        is_reg=False,
        act_fun=F.selu,
        dropout_frac=0.5,
        patience=75,
        tol=0.00001,  # Opraveno: nyní používáme hodnotu z trial
        weight_decay=0.01,
        n_epochs=300,
        neuron_layers= [200],  # Použití neuron_layers_size
        batch_size=512,
        optimizer=optim.RMSprop,
        lr=1e-5,
        random_seed=69
    )
model_final.fit(X1, y1)

In [ ]:
pred_test = model_final.predict(X3)
pred_test = pred_test > 0.5
print(matthews_corrcoef(pred_test, y3))
print(matthews_corrcoef(model_final.predict(X1) > 0.5, y1))

In [ ]:
from collections import Counter

print("Train:", Counter(y1))
print("Valid:", Counter(y2))
print("Test: ", Counter(y3))


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

# Sloučení dat
X_all = np.concatenate([X1, X2, X3])
y_all = np.concatenate([['train'] * len(X1), ['val'] * len(X2), ['test'] * len(X3)])

# t-SNE transformace
X_tsne = TSNE(n_components=2, random_state=42).fit_transform(X_all)

# Barvy: 0 = train, 1 = val, 2 = test
colors = ["green" if l == 'train' else "yellow" if l == 'val' else "red" for l in y_all]

# Vykreslení
plt.figure(figsize=(8,6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=colors, cmap='tab10', alpha=0.6)
plt.title("t-SNE: Train vs Val vs Test distribuce")
plt.xlabel("t-SNE dim 1")
plt.ylabel("t-SNE dim 2")
plt.legend(handles=scatter.legend_elements()[0], labels=['Train', 'Val', 'Test'])
plt.grid(True)
plt.show()
